In [2]:
import os
print(os.getcwd()) # 현재 작업 디렉토리 출력

/home/spai0723


# 텍스트 < 300자 (파싱실패 의심)

In [3]:
import pandas as pd

# 1. 절대 경로 설정
csv_path = '/home/shared/data_list.csv'

# 2. CSV 파일 불러오기
df = pd.read_csv(csv_path)

# 3. 모든 행의 텍스트 길이 계산 (NaN은 0으로 처리)
df['텍스트길이'] = df['텍스트'].fillna('').str.len()

# 4. 정렬 및 상위 5개 추출
# sort_values(ascending=False): 내림차순 (큰 숫자가 위로)
top_5 = df[['파일명', '텍스트길이']].sort_values(by='텍스트길이', ascending=False).head(5)

# 5. display()를 활용한 깔끔한 출력
print("--- 📋 텍스트 내용이 가장 많은 파일 TOP 5 ---")

# .style을 사용하면 표 안에 막대 그래프를 그려서 크기 비교가 쉬워집니다.
display(top_5.style.bar(subset=['텍스트길이'], color='#A9D0F5')
             .set_properties(**{'text-align': 'left'})
             .format({'텍스트길이': '{:,}'})) # 천 단위 콤마 추가

--- 📋 텍스트 내용이 가장 많은 파일 TOP 5 ---


,파일명,텍스트길이
34,한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및 1차).hwp,"18,335"
72,사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스.hwp,"18,039"
55,한국생산기술연구원_2세대 전자조달시스템 기반구축사업.hwp,"17,414"
33,울산광역시_2024년 버스정보시스템 확대 구축 및 기능개선 용역.hwp,"15,856"
21,한국보건산업진흥원_의료기기산업 종합정보시스템(정보관리기관) 기능.hwp,"9,007"


In [4]:
import pandas as pd
import os

# 1. 수빈님의 작업 폴더 절대 경로 설정
# /home/spai0723 위치에 계시므로, 그 아래 Bidcoin 폴더를 명시합니다.
BASE_DIR = '/home/shared'
CSV_PATH = os.path.join(BASE_DIR, 'data_list.csv')
FILES_DIR = os.path.join(BASE_DIR, 'files')

# 2. CSV 파일 불러오기
try:
    df = pd.read_csv(CSV_PATH)
    print("✅ CSV 파일을 성공적으로 불러왔습니다.")
except FileNotFoundError:
    print(f"❌ 에러: {CSV_PATH} 파일을 찾을 수 없습니다. 파일명을 확인해 주세요.")

# 3. 실제 폴더 내 파일 리스트 가져오기
if os.path.exists(FILES_DIR):
    actual_files = os.listdir(FILES_DIR)
    print(f"✅ 실제 파일 폴더({FILES_DIR})를 찾았습니다.")
else:
    actual_files = []
    print(f"❌ 에러: {FILES_DIR} 폴더를 찾을 수 없습니다.")

# 4. 데이터 분석 및 필터링
# CSV 내 파일명이 실제 폴더에 존재하는지 확인
df['파일존재여부'] = df['파일명'].apply(lambda x: x in actual_files)

# 텍스트가 비어있을(NaN) 경우를 대비해 빈 문자열로 채운 뒤 길이를 잽니다.
df['텍스트길이'] = df['텍스트'].fillna('').str.len()

# 폴더에 파일은 실존하지만, 텍스트가 300자 미만인 것들만 추출
recovery_needed = df[(df['파일존재여부'] == True) & (df['텍스트길이'] < 300)]

print("-" * 50)
print(f"📊 전체 데이터(CSV): {len(df)}건")
print(f"📂 폴더 내 실제 파일: {len(actual_files)}개")
print(f"⚠️ 텍스트 복구가 필요한 파일: {len(recovery_needed)}개")
print("-" * 50)

if not recovery_needed.empty:
    print("\n📋 복구 필요 파일 전체 목록")
    #print(recovery_needed[['파일명', '텍스트길이']])
    display(recovery_needed[['파일명', '텍스트길이']])

✅ CSV 파일을 성공적으로 불러왔습니다.
✅ 실제 파일 폴더(/home/shared/files)를 찾았습니다.
--------------------------------------------------
📊 전체 데이터(CSV): 100건
📂 폴더 내 실제 파일: 101개
⚠️ 텍스트 복구가 필요한 파일: 6개
--------------------------------------------------

📋 복구 필요 파일 전체 목록


,파일명,텍스트길이
2,한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp,234
8,재단법인스포츠윤리센터_스포츠윤리센터 LMS(학습지원시스템) 기능개선.hwp,298
12,서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf,220
17,2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp,186
18,한국발명진흥회 입찰공고_2024년 건설기술에 관한 특허·실용신안 활용실.hwp,89
20,전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp,130


In [24]:
import fitz  # PyMuPDF
import re    # 정규표현식

pdf_file = "/home/shared/files/서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf"

doc = fitz.open(pdf_file)
full_text = ""
for page in doc:
    full_text += page.get_text()

print(f"🧹 [정제 전] 텍스트 길이: {len(full_text)}")

# 1. 1차 정제: 불필요한 특수기호 제거 (마지막에 \@ 추가됨!)
cleaned_text = re.sub(r'[^가-힣a-zA-Z0-9\s\.\(\)\[\]\/\,\%\:\-\·\?\!\@]', ' ', full_text)

# 2. 2차 정제: 목차 점선 제거 (마침표나 가운데 점이 2개 이상 연속될 경우 공백으로 치환)
cleaned_text = re.sub(r'[\.·]{2,}', ' ', cleaned_text)

# 3. 3차 정제: 다중 공백을 하나의 공백으로 압축하고 양끝 여백 제거
cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()

print(f"✨ [정제 후] 텍스트 길이: {len(cleaned_text)}")
print("-" * 60)
print(cleaned_text[:500])

🧹 [정제 전] 텍스트 길이: 116224
✨ [정제 후] 텍스트 길이: 104123
------------------------------------------------------------
[사전공개용] 제 안 요 청 서 본 제안요청서는 입찰참여의 균등한 기회 제공을 위해 규격을 공개하기 위한 자료로써 실제 입찰공고 시 사업금액, 과업내용, 평가항목, 제출서류 등은 변경될 수 있으니 반드시 확인하시기 바랍니다. 2023. 06. 담당 성명 소 속 전화번호 e-mail 이석준 서울시립대학교 (입학처) 02-6490-6176 lsjptrs@uos.ac.kr 사 업 명 학업성취도 다차원 종단분석 통합시스템 1차 고도화 주관기관 서 울 시 립 대 학 교 입 학 처 목 차 . 사업안내 1. 사업개요 01 2. 추진배경 및 필요성 01 3. 사업근거 및 3개년 추진계획 01 4. 사업범위 02 5. 기대효과 03 . 대상업무 현황 1. 기존 연계 현황 03 2. 개발 완료 내역 04 3. 추가 연계 필요 현황 06 4. 시스템 현황 06 . 사업추진 방안 1. 추진체계 07 2. 추진일정 08 3. 추진방안 09 . 제안요청내용 1. 시스템 개발 범위 09 2. 제안요청 내용 1


In [25]:
# 1. 타겟 파일명 지정
target_file_name = "서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf"

# 2. 데이터프레임(df)에서 해당 파일의 행(Index) 찾기
idx = df[df['파일명'] == target_file_name].index

if not idx.empty:
    # 3. 정제된 텍스트와 새로운 텍스트 길이를 df에 덮어쓰기 (Update)
    df.loc[idx[0], '텍스트'] = cleaned_text
    df.loc[idx[0], '텍스트길이'] = len(cleaned_text)
    
    print(f"✅ 업데이트 완료: {target_file_name}")
    print(f"✅ 현재 df에 저장된 길이: {len(df.loc[idx[0], '텍스트'])}자")
else:
    print("❌ 에러: 데이터프레임에서 파일을 찾을 수 없습니다.")

# 4. 최종 결과 마스터 CSV 파일에 영구 저장하기 (잠그기!)
save_path = '/home/spai0723/Bidcoin/bid_master_cleaned.csv'
df.to_csv(save_path, index=False, encoding='utf-8-sig')
print(f"💾 텍스트 정제 결과가 '{save_path}'에 안전하게 저장되었습니다!")

✅ 업데이트 완료: 서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf
✅ 현재 df에 저장된 길이: 104123자
💾 텍스트 정제 결과가 '/home/spai0723/Bidcoin/bid_master_cleaned.csv'에 안전하게 저장되었습니다!


In [6]:
import olefile
import zlib
import re

def get_hwp_text_final_exorcism(file_path):
    # 1. 상용 한글 2,350자 세트 (가장 많이 쓰이는 글자들만 정의)
    # 이 리스트에 없는 '샇', '엀', '딀' 같은 글자는 무조건 탈락입니다.
    def is_clean_hangul(char):
        if not '가' <= char <= '힣': return True # 한글 아니면 통과 (영어/숫자/기호)
        
        # CP949 중에서도 '현대 한국어 2,350자' 범위만 체크
        # (이 방법이 가장 확실하게 노이즈를 거릅니다)
        try:
            code = char.encode('cp949')
            # CP949에서 한글은 첫 바이트가 0xB0 이상입니다.
            # 0xC8 이 넘어가면 실생활에서 거의 안 쓰는 글자들이 많아집니다.
            return 0xB0 <= code[0] <= 0xC8
        except:
            return False

    try:
        # --- 추출 로직 ---
        f = olefile.OleFileIO(file_path)
        dirs = f.listdir()
        bodytext_sections = [d for d in dirs if 'BodyText' in d]
        
        raw_text = ""
        for section in bodytext_sections:
            data = f.openstream(section).read()
            decompressed = zlib.decompress(data, -15)
            raw_text += decompressed.decode('utf-16', errors='ignore')
        
        # --- 정제 로직 ---
        # 1) 이미지 관련 정보 강제 삭제
        text = re.sub(r'원본 그림의 이름:.*?pixel', ' ', raw_text, flags=re.DOTALL)
        text = re.sub(r'가로 \d+pixel, 세로 \d+pixel', ' ', text)
        
        # 2) 기본 기호만 남기기
        text = re.sub(r'[^가-힣a-zA-Z0-9\s\.\(\)\[\]\/\,\%\:\-\·\?\!]', ' ', text)
        
        tokens = text.split()
        clean_tokens = []
        
        for t in tokens:
            # 3) 단어 내의 모든 글자가 '깨끗한 한글'인지 전수 조사
            if all(is_clean_hangul(c) for c in t):
                # 4) 1글자 노이즈 추가 차단 (조사/숫자 아니면 삭제)
                if len(t) == 1 and t not in '이가을를에와과도한1234567890o-·':
                    continue
                # 5) 한글-영어 혼합 노이즈 삭제 (ex: w건설)
                if re.search(r'[a-zA-Z][가-힣]', t) or re.search(r'[가-힣][a-zA-Z]', t):
                    continue
                clean_tokens.append(t)
        
        return " ".join(clean_tokens)

    except Exception as e:
        return f"❌ 엑소시즘 실패: {str(e)}"

# --- 실행 ---
target_file = "/home/shared/files/한국발명진흥회 입찰공고_2024년 건설기술에 관한 특허·실용신안 활용실.hwp"
final_text = get_hwp_text_final_exorcism(target_file)

print(f"✨ 엑소시즘 완료 (길이: {len(final_text)}자)")
print("-" * 50)
print(final_text[:1000])

✨ 엑소시즘 완료 (길이: 27835자)
--------------------------------------------------
0 4 8 2024년 건설기술에 관한 특허·실용신안 활용실적 관리시스템 개편 용역 제안요청서 부서명 6 연락처 가 박수희 가 7 7 7 7 7 7 5 o - 가 5뀀 [저 뷕솬 2 추진 배경 필요성 - 효율적인 서비스 운영을 위한 시스템 개편 추진 - o 최근 3년간 평균 46%의 심사의뢰건 증가로 인해 심사 효율성 제고 시급 o 심사 프로세스 간소화를 통한 업무처리 능력 향상 o 관리자용 시스템에서 업무처리를 위한 기능 개선 o 사용자 중심의 직관적인 인터페이스 레이아웃 개선 - 이용 편리성 증대를 위한 사용자 인터페이스 개선 o 사용자 중심의 직관적인 인터페이스 레이아웃 필요성 제고 o 민원발생 소지를 낮추기 위한 시스템 개선 o 기관 시스템과 정보 연계 체계 구축 - - 세로 491pixel 관계기관 국토교통부 기술정책과 특허청 건설기술심사과 뀀견 섀타 가 가 가 5 가 가 저· 가 가 가 가 업무 현황 세로 64pixel 시스템 현황 시스템 구성도 추진 목표 o 건설기술에 관한 특허·실용신안 활용실적 관리시스템 개편을 통해 대민용 정보 제공기능 강화 사업 담당자의 성과관리 사업 운영 강화 5 o 기존 시스템의 사용 기능 개선 사용자 중심의 직관적인 인터페이스 레이아웃 설계를 통해 고객 만족도 향상 추진체계 0 o 추진 조직도 7 사업총괄 3 7씀- 7 협력 부서 기업성장지원실 용역수행업체 외부 전문업체 3 4 구분 주요 역할 기업성장지원실 협력 부서 사업 추진 의견제시 협의·조정 외부 용역사업자 가 구분 2 o 요구사항 분석 툀팀 툀팀 툀팀 퐀팀 븀팀 섀팀 섀팀 섀팀 사용자 시스템 재구축 씀팀 가 · 묀팀 · 혀팀 심사위원용 시스템 구축 케팀 턀팀 쨀팀 쨀팀 숀팀 쨀팀 구축 완료 사용자 교육(활용) 먀팀 먀팀 먀팀 2 주요 과업내용 o 효율적인 업무처리 지원을 위한 기능 구축 0 - 신청/심사 프로세스 간소화 자료 업/다운

In [7]:
import os
import olefile
import zlib
import re
import pandas as pd

# 1. 수빈님의 최종 병기: 상용 한글 2,350자 필터링 함수
def get_hwp_text_final_exorcism(file_path):
    def is_clean_hangul(char):
        if not '가' <= char <= '힣': return True
        try:
            code = char.encode('cp949')
            return 0xB0 <= code[0] <= 0xC8
        except: return False

    try:
        f = olefile.OleFileIO(file_path)
        dirs = f.listdir()
        bodytext_sections = [d for d in dirs if 'BodyText' in d]
        raw_text = ""
        for section in bodytext_sections:
            data = f.openstream(section).read()
            decompressed = zlib.decompress(data, -15)
            raw_text += decompressed.decode('utf-16', errors='ignore')
        
        # 이미지 정보 및 불필요한 메타데이터 제거
        text = re.sub(r'원본 그림의 이름:.*?pixel', ' ', raw_text, flags=re.DOTALL)
        text = re.sub(r'가로 \d+pixel, 세로 \d+pixel', ' ', text)
        text = re.sub(r'[^가-힣a-zA-Z0-9\s\.\(\)\[\]\/\,\%\:\-\·\?\!]', ' ', text)
        
        tokens = text.split()
        clean_tokens = []
        for t in tokens:
            if all(is_clean_hangul(c) for c in t):
                if len(t) == 1 and t not in '이가을를에와과도한1234567890o-·':
                    continue
                if re.search(r'[a-zA-Z][가-힣]', t) or re.search(r'[가-힣][a-zA-Z]', t):
                    continue
                clean_tokens.append(t)
        return " ".join(clean_tokens)
    except:
        return None

# 2. 남은 4개 파일 리스트 설정
remaining_files = [
    "한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp",
    "재단법인스포츠윤리센터_스포츠윤리센터 LMS(학습지원시스템) 기능개선.hwp",
    "25 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp",
    "전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp"
]

print("📦 남은 4개 파일의 복구를 시작합니다...")
print("-" * 40)

# 3. 자동 복구 루프 실행
for file_name in remaining_files:
    # 파일 절대 경로 생성
    file_path = os.path.join("/home/shared/files/", file_name)
    
    # 해당 파일명이 데이터프레임에 있는지 확인
    idx_list = df[df['파일명'] == file_name].index
    
    if not idx_list.empty:
        idx = idx_list[0]
        print(f"🔍 처리 중: {file_name}")
        
        restored_text = get_hwp_text_final_exorcism(file_path)
        
        if restored_text and len(restored_text) > 100:
            df.loc[idx, '텍스트'] = restored_text
            df.loc[idx, '텍스트길이'] = len(restored_text)
            print(f"   ✅ 복구 성공! (최종 길이: {len(restored_text):,}자)")
        else:
            print(f"   ⚠️ 주의: 텍스트가 너무 짧거나 추출에 실패했습니다.")
    else:
        print(f"   ❌ 오류: 데이터프레임에서 '{file_name}'을 찾을 수 없습니다.")

print("-" * 40)
print("✨ 모든 파일의 복구 및 업데이트가 완료되었습니다!")

# 4. 최종 결과 확인
display(df[df['파일명'].isin(remaining_files)][['파일명', '텍스트길이']])

📦 남은 4개 파일의 복구를 시작합니다...
----------------------------------------
🔍 처리 중: 한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp
   ✅ 복구 성공! (최종 길이: 54,439자)
🔍 처리 중: 재단법인스포츠윤리센터_스포츠윤리센터 LMS(학습지원시스템) 기능개선.hwp
   ✅ 복구 성공! (최종 길이: 35,327자)
   ❌ 오류: 데이터프레임에서 '25 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp'을 찾을 수 없습니다.
🔍 처리 중: 전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp
   ✅ 복구 성공! (최종 길이: 34,048자)
----------------------------------------
✨ 모든 파일의 복구 및 업데이트가 완료되었습니다!


,파일명,텍스트길이
2,한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp,54439
8,재단법인스포츠윤리센터_스포츠윤리센터 LMS(학습지원시스템) 기능개선.hwp,35327
20,전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp,34048


In [8]:
# 1. '구미'라는 단어가 들어간 파일명을 직접 찾아서 확인해보기
gumi_check = df[df['파일명'].str.contains('구미', na=False)]
display(gumi_check[['파일명']])

# 2. 확인된 진짜 이름을 복구 리스트에 넣거나, 아래처럼 '포함' 방식으로 루프 수정
remaining_files = [
    "한국생산기술연구원", 
    "스포츠윤리센터", 
    "구미", # '구미'만 들어가면 찾도록 핵심 단어만 배치
    "전북대학교"
]

print("📦 키워드 기반으로 다시 검색하여 복구를 시작합니다...")

for keyword in remaining_files:
    # 핵심 단어가 포함된 행을 찾습니다.
    idx_list = df[df['파일명'].str.contains(keyword, na=False)].index
    
    if not idx_list.empty:
        idx = idx_list[0]
        file_name = df.loc[idx, '파일명'] # 실제 저장된 전체 파일명 가져오기
        file_path = os.path.join("/home/shared/files/", file_name)
        
        print(f"🔍 발견 및 처리 중: {file_name}")
        
        restored_text = get_hwp_text_final_exorcism(file_path)
        if restored_text:
            df.loc[idx, '텍스트'] = restored_text
            df.loc[idx, '텍스트길이'] = len(restored_text)
            print(f"   ✅ 복구 성공! (길이: {len(restored_text):,}자)")
    else:
        print(f"   ❌ 키워드 '{keyword}'(을)를 포함한 파일을 찾지 못했습니다.")

,파일명
17,2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp


📦 키워드 기반으로 다시 검색하여 복구를 시작합니다...
🔍 발견 및 처리 중: 한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp
   ✅ 복구 성공! (길이: 54,439자)
🔍 발견 및 처리 중: 재단법인스포츠윤리센터_스포츠윤리센터 LMS(학습지원시스템) 기능개선.hwp
   ✅ 복구 성공! (길이: 35,327자)
🔍 발견 및 처리 중: 2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp
   ✅ 복구 성공! (길이: 26,973자)
🔍 발견 및 처리 중: 전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp
   ✅ 복구 성공! (길이: 34,048자)


In [9]:
df.to_csv('/home/spai0723/Bidcoin/data_list_final_master.csv', index=False, encoding='utf-8-sig')

In [10]:
# 1. 엑소시즘 함수로 2만 자 텍스트 다시 뽑기
target_file_name = "한국발명진흥회 입찰공고_2024년 건설기술에 관한 특허·실용신안 활용실.hwp"
target_path = "/home/shared/files/" + target_file_name

# 추출하기
restored_text = get_hwp_text_final_exorcism(target_path)

# 2. 데이터프레임(df)의 해당 파일 행(Index)에 '직접' 업데이트하기
# 파일명이 일치하는 행의 index를 찾아서 텍스트와 길이를 덮어씌웁니다.
idx = df[df['파일명'] == target_file_name].index

if not idx.empty:
    df.loc[idx[0], '텍스트'] = restored_text
    df.loc[idx[0], '텍스트길이'] = len(restored_text)
    print(f"✅ 업데이트 완료: {target_file_name}")
    print(f"✅ 현재 df에 저장된 길이: {len(df.loc[idx[0], '텍스트'])}자")
else:
    print("❌ 에러: 데이터프레임에서 파일을 찾을 수 없습니다.")

# 3. 중요!! 내 개인 폴더에 CSV 파일로 영구 저장하기 (다음에 불러올 때 89자로 안 돌아가게)
# 이 작업을 안 하면 주피터 재시작 시 다시 89자로 돌아갑니다!
df.to_csv('/home/spai0723/Bidcoin/bid_master_cleaned.csv', index=False, encoding='utf-8-sig')
print("💾 최종 결과가 'bid_master_cleaned.csv'에 안전하게 저장되었습니다!")

✅ 업데이트 완료: 한국발명진흥회 입찰공고_2024년 건설기술에 관한 특허·실용신안 활용실.hwp
✅ 현재 df에 저장된 길이: 27835자
💾 최종 결과가 'bid_master_cleaned.csv'에 안전하게 저장되었습니다!


# 텍스트 < 요약 (이상 역전)

In [11]:
import pandas as pd

# 1. 검사 대상 키워드 설정 [cite: 2026-03-09]
check_keywords = ['구미', '건설기술', 'JST']

# 2. 진단 결과를 담을 리스트
diagnosis_results = []

print("📊 대상 파일들에 대한 '텍스트 vs 요약' 길이 비교 검사를 시작합니다...")

# 요약 컬럼의 진짜 이름 자동 찾기 (요약 길이가 0으로 나오는 문제 방지) [cite: 2026-03-09]
summary_col = [c for c in df.columns if '요약' in c or 'summary' in c.lower()]
col_name = summary_col[0] if summary_col else '요약'

for kw in check_keywords:
    # 키워드가 포함된 행 찾기 [cite: 2026-03-09]
    target_rows = df[df['파일명'].str.contains(kw, na=False)]
    
    for _, row in target_rows.iterrows():
        text_content = str(row['텍스트']) if pd.notna(row['텍스트']) else ""
        summary_content = str(row[col_name]) if col_name in df.columns and pd.notna(row[col_name]) else ""
        
        text_len = len(text_content)
        summary_len = len(summary_content)
        
        # 텍스트가 요약보다 짧거나, 본문이 너무 짧은 경우(89자 등) 위험으로 표시 [cite: 2026-03-09]
        is_inverted = text_len < summary_len or text_len < 300
        
        diagnosis_results.append({
            "파일명": row['파일명'][:40] + "...", 
            "텍스트 길이": text_len,
            "요약 길이": summary_len,
            "역전 여부": "⚠️ 역전(위험)" if is_inverted else "✅ 정상"
        })

# 3. 결과 출력
diagnosis_df = pd.DataFrame(diagnosis_results)
display(diagnosis_df)

# 4. 종합 요약 출력 [cite: 2026-03-09]
inverted_count = len(diagnosis_df[diagnosis_df['역전 여부'] == "⚠️ 역전(위험)"])
if inverted_count > 0:
    print(f"\n📢 진단 결과: {inverted_count}개의 파일에서 아직 역전 현상이나 본문 부족이 발견되었습니다.")
else:
    print("\n🎉 진단 결과: 모든 파일의 텍스트가 정상 범위 내에 있습니다!")

📊 대상 파일들에 대한 '텍스트 vs 요약' 길이 비교 검사를 시작합니다...


,파일명,텍스트 길이,요약 길이,역전 여부
0,2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경...,26973,212,✅ 정상
1,한국발명진흥회 입찰공고_2024년 건설기술에 관한 특허·실용신안 활용실....,27835,307,✅ 정상
2,전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp...,34048,288,✅ 정상



🎉 진단 결과: 모든 파일의 텍스트가 정상 범위 내에 있습니다!


# 공고번호 누락

In [14]:
print("현재 데이터프레임의 컬럼 목록:")
print(df.columns.tolist())

현재 데이터프레임의 컬럼 목록:
['공고 번호', '공고 차수', '사업명', '사업 금액', '발주 기관', '공개 일자', '입찰 참여 시작일', '입찰 참여 마감일', '사업 요약', '파일형식', '파일명', '텍스트', '파일존재여부', '텍스트길이']


In [17]:
import pandas as pd

# 1. 컬럼명 공백 제거 (안전장치)
df.columns = df.columns.str.strip()

# 2. '공고 번호'가 없는 데이터만 필터링 [cite: 2026-03-09]
missing_id_df = df[df['공고 번호'].isna()]

# 3. 전체 목록 출력 (사업명과 발주 기관이 잘 있는지 확인용) [cite: 2026-03-09]
print(f"📋 공고 번호 누락 데이터: 총 {len(missing_id_df)}건")
print("-" * 60)

# 행 제한 없이 모든 목록을 보여줍니다.
pd.set_option('display.max_rows', None)
display(missing_id_df[['파일명', '사업명', '발주 기관', '공개 일자']])
pd.reset_option('display.max_rows')

📋 공고 번호 누락 데이터: 총 18건
------------------------------------------------------------


,파일명,사업명,발주 기관,공개 일자
7,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,차세대 포털·학사 정보시스템 구축사업,고려대학교,2024-07-01 00:00:00
12,서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf,[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차 고도화 용역,서울시립대학교,2023-06-20 00:00:00
13,경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp,[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정,경희대학교,2024-05-02 00:00:00
14,한국수자원공사_건설통합시스템(CMS) 고도화.hwp,건설통합시스템(CMS) 고도화,한국수자원공사,2024-05-31 00:00:00
15,국가과학기술지식정보서비스_통합정보시스템 고도화 용역.hwp,통합정보시스템 고도화 용역,국가과학기술지식정보서비스,2024-05-30 00:00:00
16,한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp,예약발매시스템 개량 ISMP 용역,한국철도공사 (용역),2024-05-31 00:00:00
18,한국발명진흥회 입찰공고_2024년 건설기술에 관한 특허·실용신안 활용실.hwp,2024년 건설기술에 관한 특허·실용신안 활용실적 관리시스템 개편 용역,한국발명진흥회 입찰공고,2024-06-07 00:00:00
24,한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp,용인 첨단 시스템반도체 국가산단 용수공급사업 타당성조사 및 기본계획 수립 용역,한국수자원공사,2024-05-13 00:00:00
26,KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp,[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 국회 방송시스템 구축 ...,KOICA 전자조달,2024-10-24 00:00:00
27,한국수자원공사_수도사업장 통합 사고분석솔루션 시범구축 용역.hwp,수도사업장 통합 사고분석솔루션 시범구축 용역,한국수자원공사,2024-05-07 00:00:00


In [18]:
# 1. 아까 만든 missing_id_df에서 결측치가 있던 18건의 행 번호(Index)만 쏙 뽑아 기억해둡니다.
target_indices = missing_id_df.index

# 2. 원본 데이터프레임(df)에 공고 번호 업데이트 실행 (사업명_발주 기관)
df['공고 번호'] = df.apply(
    lambda row: f"{row['사업명']}_{row['발주 기관']}" if pd.isna(row['공고 번호']) else row['공고 번호'],
    axis=1
)

# 3. 전체 결측치가 0이 되었는지 점검
after_missing_id = df['공고 번호'].isna().sum()
print(f"🔹 남은 공고 번호 결측치: {after_missing_id}건")
print("-" * 60)

# 4. 업데이트 결과 확인 (수동 번호 입력 대신 target_indices 변수 사용!)
print(f"✨ 업데이트 된 {len(target_indices)}건의 데이터 확인:")

# display.max_rows를 풀어서 18건을 한 번에 다 봅니다.
pd.set_option('display.max_rows', None)
display(df.loc[target_indices, ['사업명', '발주 기관', '공고 번호']])
pd.reset_option('display.max_rows')

🔹 남은 공고 번호 결측치: 0건
------------------------------------------------------------
✨ 업데이트 된 18건의 데이터 확인:


,사업명,발주 기관,공고 번호
7,차세대 포털·학사 정보시스템 구축사업,고려대학교,차세대 포털·학사 정보시스템 구축사업 _고려대학교
12,[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차 고도화 용역,서울시립대학교,[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차 고도화 용역_서울시립대학교
13,[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정,경희대학교,[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정_경희대학교
14,건설통합시스템(CMS) 고도화,한국수자원공사,건설통합시스템(CMS) 고도화_한국수자원공사
15,통합정보시스템 고도화 용역,국가과학기술지식정보서비스,통합정보시스템 고도화 용역_국가과학기술지식정보서비스
16,예약발매시스템 개량 ISMP 용역,한국철도공사 (용역),예약발매시스템 개량 ISMP 용역_한국철도공사 (용역)
18,2024년 건설기술에 관한 특허·실용신안 활용실적 관리시스템 개편 용역,한국발명진흥회 입찰공고,2024년 건설기술에 관한 특허·실용신안 활용실적 관리시스템 개편 용역_한국발명진흥...
24,용인 첨단 시스템반도체 국가산단 용수공급사업 타당성조사 및 기본계획 수립 용역,한국수자원공사,용인 첨단 시스템반도체 국가산단 용수공급사업 타당성조사 및 기본계획 수립 용역_한국...
26,[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 국회 방송시스템 구축 ...,KOICA 전자조달,[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 국회 방송시스템 구축 ...
27,수도사업장 통합 사고분석솔루션 시범구축 용역,한국수자원공사,수도사업장 통합 사고분석솔루션 시범구축 용역_한국수자원공사


# 입찰시작일 누락

In [19]:
import pandas as pd

# 1. 컬럼명 공백 제거 (안전장치)
df.columns = df.columns.str.strip()

# 2. '입찰 참여 시작일'이 없는 데이터만 필터링
missing_date_df = df[df['입찰 참여 시작일'].isna()]

# 3. 전체 목록 출력 (공개 일자가 잘 있는지 확인용)
print(f"📋 입찰 참여 시작일 누락 데이터: 총 {len(missing_date_df)}건")
print("-" * 60)

# 행 제한 없이 모든 목록을 보여줍니다.
pd.set_option('display.max_rows', None)
display(missing_date_df[['파일명', '사업명', '발주 기관', '공개 일자', '입찰 참여 시작일']])
pd.reset_option('display.max_rows')

📋 입찰 참여 시작일 누락 데이터: 총 26건
------------------------------------------------------------


,파일명,사업명,발주 기관,공개 일자,입찰 참여 시작일
0,한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp,한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화,한영대학,2024-10-04 13:51:23,NaN
12,서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf,[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차 고도화 용역,서울시립대학교,2023-06-20 00:00:00,NaN
13,경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp,[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정,경희대학교,2024-05-02 00:00:00,NaN
14,한국수자원공사_건설통합시스템(CMS) 고도화.hwp,건설통합시스템(CMS) 고도화,한국수자원공사,2024-05-31 00:00:00,NaN
17,2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp,2025 구미아시아육상경기선수권대회 종합정보시스템 및 홈페이지 등 구축 용역,2025 구미 아시아육상경기선수권대회 조직위원회,2024-09-10 16:31:49,NaN
23,수협중앙회_강릉어선안전조업국 상황관제시스템 구축.hwp,강릉어선안전조업국 상황관제시스템 구축,수협중앙회,2025-02-11 10:27:38,NaN
24,한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp,용인 첨단 시스템반도체 국가산단 용수공급사업 타당성조사 및 기본계획 수립 용역,한국수자원공사,2024-05-13 00:00:00,NaN
26,KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp,[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 국회 방송시스템 구축 ...,KOICA 전자조달,2024-10-24 00:00:00,NaN
31,인천광역시 동구_수도국산달동네박물관 전시해설 시스템 구축(협상에 .hwp,수도국산달동네박물관 전시해설 시스템 구축(협상에 의한 계약),인천광역시 동구,2025-01-24 19:56:15,NaN
38,광주과학기술원_학사시스템 기능개선 사업.hwp,학사시스템 기능개선 사업,광주과학기술원,2024-12-09 08:52:59,NaN


In [20]:
# 1. 아까 확인한 26건의 행 번호(Index)를 그대로 활용합니다.
target_indices = missing_date_df.index

# 2. 원본 데이터프레임(df)의 빈칸을 '공개 일자'로 덮어씌웁니다. (메모리에서만!)
df['입찰 참여 시작일'] = df['입찰 참여 시작일'].fillna(df['공개 일자'])

# 3. 전체 결측치가 0이 되었는지 점검
after_missing_date = df['입찰 참여 시작일'].isna().sum()
print(f"🔹 남은 입찰 참여 시작일 결측치: {after_missing_date}건")
print("-" * 60)

# 4. 업데이트 결과 확인 (저장 없이 화면 출력만 진행)
print(f"✨ 업데이트 된 {len(target_indices)}건의 데이터 확인:")

# display.max_rows를 풀어서 26건을 한 번에 다 보여줍니다.
pd.set_option('display.max_rows', None)
# 비교하기 쉽게 '공개 일자'와 '입찰 참여 시작일'을 나란히 출력합니다.
display(df.loc[target_indices, ['파일명', '공개 일자', '입찰 참여 시작일']])
pd.reset_option('display.max_rows')

🔹 남은 입찰 참여 시작일 결측치: 0건
------------------------------------------------------------
✨ 업데이트 된 26건의 데이터 확인:


,파일명,공개 일자,입찰 참여 시작일
0,한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp,2024-10-04 13:51:23,2024-10-04 13:51:23
12,서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf,2023-06-20 00:00:00,2023-06-20 00:00:00
13,경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp,2024-05-02 00:00:00,2024-05-02 00:00:00
14,한국수자원공사_건설통합시스템(CMS) 고도화.hwp,2024-05-31 00:00:00,2024-05-31 00:00:00
17,2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp,2024-09-10 16:31:49,2024-09-10 16:31:49
23,수협중앙회_강릉어선안전조업국 상황관제시스템 구축.hwp,2025-02-11 10:27:38,2025-02-11 10:27:38
24,한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp,2024-05-13 00:00:00,2024-05-13 00:00:00
26,KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp,2024-10-24 00:00:00,2024-10-24 00:00:00
31,인천광역시 동구_수도국산달동네박물관 전시해설 시스템 구축(협상에 .hwp,2025-01-24 19:56:15,2025-01-24 19:56:15
38,광주과학기술원_학사시스템 기능개선 사업.hwp,2024-12-09 08:52:59,2024-12-09 08:52:59


In [21]:
# 1. 덮어쓰기 완료된 데이터프레임을 수빈님의 개인 폴더에 CSV 파일로 영구 저장합니다.
save_path = '/home/spai0723/Bidcoin/bid_master_cleaned.csv'
df.to_csv(save_path, index=False, encoding='utf-8-sig')

# 2. 완료 메시지 출력
print("🎉 축하합니다! 공고 번호와 입찰 참여 시작일 결측치가 모두 해결되었습니다.")
print(f"💾 완벽해진 최종 마스터 데이터가 '{save_path}'에 안전하게 저장되었습니다!")

🎉 축하합니다! 공고 번호와 입찰 참여 시작일 결측치가 모두 해결되었습니다.
💾 완벽해진 최종 마스터 데이터가 '/home/spai0723/Bidcoin/bid_master_cleaned.csv'에 안전하게 저장되었습니다!
